# RAMP Demand Aggregation — Norte Amazonia Bolivia

This notebook aggregates the RAMP simulation outputs into annual electricity demand (GWh/year) per cluster, formatted as EnergyScope input files (`Demands_C*.csv`).

For each cluster, each municipality's `load_curve_energy_service_full_year_Norte_Amazonia.csv` is read, mapped to EnergyScope end-use categories, and summed across all municipalities in the cluster.

In [6]:
import os
import pandas as pd

OUTPUT_DIR = "output_energyscope"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir", "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando", "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# Maps each RAMP column → (EnergyScope sector, end-use category)
MAPPING = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    "big_school_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "big_school_ICT":                      ("SERVICES",   "ELECTRICITY"),
    "big_school_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "big_school_space_cooling":            ("SERVICES",   "SPACE_COOLING"),
    "health_center_illumination":          ("SERVICES",   "LIGHTING_R_C"),
    "health_center_ICT":                   ("SERVICES",   "ELECTRICITY"),
    "health_center_cold_storage":          ("SERVICES",   "FOOD_PRESERVATION"),
    "health_center_space_cooling":         ("SERVICES",   "SPACE_COOLING"),
    "health_center_water_heating":         ("SERVICES",   "HEAT_LOW_T_HW"),
    "health_center_water_supply":          ("SERVICES",   "ELECTRICITY"),
    "health_center_medical_equip":         ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
    "public_lighting_illumination":        ("PUBLIC_LIGHTING", "LIGHTING_P"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
}

## 1. Setup — clusters and RAMP → EnergyScope mapping

## 2. Helper functions

- **`create_empty_demands()`** — builds the blank EnergyScope demand table (21 rows, all sectors at 0.0), matching the `Demands.csv` format exactly
- **`compute_municipality(muni_name)`** — reads a municipality's RAMP output, maps each column through `MAPPING`, and returns annual GWh per `(sector, end_use)` pair

> Unit conversion: watts × minutes → GWh, i.e. divide by `60 × 10⁹`  
> (RAMP outputs power in W at 1-minute timesteps over 1 year = 525,600 minutes)

In [7]:
def create_empty_demands():
    """Build the empty EnergyScope demand table (21 rows, all sectors at 0.0)."""
    columns = [
        "Category", "Subcategory", "parameter name",
        "HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
        "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS",
        "Units"
    ]
    rows = [
        ["Electricity",  "Electricity",                            "ELECTRICITY",                   "[GWh]"],
        ["Lighting",     "Building lighting",                      "LIGHTING_R_C",                  "[GWh]"],
        ["Lighting",     "Public lighting",                        "LIGHTING_P",                    "[GWh]"],
        ["Heat",         "High temperature",                       "HEAT_HIGH_T",                   "[GWh]"],
        ["Heat",         "Space heating",                          "HEAT_LOW_T_SH",                 "[GWh]"],
        ["Heat",         "Hot water",                              "HEAT_LOW_T_HW",                 "[GWh]"],
        ["Heat",         "Cooking",                                "COOKING",                       "[GWh]"],
        ["Cold",         "Process cooling",                        "PROCESS_COOLING",               "[GWh]"],
        ["Cold",         "Space cooling",                          "SPACE_COOLING",                 "[GWh]"],
        ["Cold",         "Food preservation",                      "FOOD_PRESERVATION",             "[GWh]"],
        ["Mobility",     "Passenger",                              "MOBILITY_PASSENGER",            "[Mpkm]"],
        ["Mobility",     "Freight",                                "MOBILITY_FREIGHT",              "[Mtkm]"],
        ["Mobility",     "Long-haul passenger flights",            "AVIATION_LONG_HAUL",            "[Mpkm]"],
        ["Mobility",     "International shipping",                 "SHIPPING",                      "[Mtkm]"],
        ["Mechanical",   "Mechanical energy commercial",           "MECHANICAL_ENERGY_COMM",        "[GWh]"],
        ["Mechanical",   "Mechanical energy industrial",           "MECHANICAL_ENERGY_IND",         "[GWh]"],
        ["Mechanical",   "Mechanical energy agriculture mobility", "MECHANICAL_ENERGY_MOV_AGR",     "[GWh]"],
        ["Mechanical",   "Mechanical energy agriculture fixed",    "MECHANICAL_ENERGY_FIX_AGR",     "[GWh]"],
        ["Mechanical",   "Mechanical energy mining",               "MECHANICAL_ENERGY_MIN",         "[GWh]"],
        ["Mechanical",   "Mechanical energy fishing",              "MECHANICAL_ENERGY_FISH_OTHERS", "[GWh]"],
        ["Non-energy",   "Non-energy",                             "NON_ENERGY",                    "[GWh]"],
    ]
    df = pd.DataFrame(columns=columns)
    for i, row in enumerate(rows):
        df.loc[i] = [row[0], row[1], row[2], 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, row[3]]
    return df


def compute_municipality(muni_name):
    """Read a municipality's RAMP CSV and return {(sector, end_use): annual_GWh}."""
    path = f"data ramp total sufficiency/{muni_name}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
    if not os.path.exists(path):
        print(f"Warning: file not found for '{muni_name}'")
        return {}
    df = pd.read_csv(path)
    results = {}
    for col in df.columns:
        if col in MAPPING:
            key = MAPPING[col]
            annual_gwh = df[col].fillna(0).sum() / 60_000_000_000.0
            results[key] = results.get(key, 0.0) + annual_gwh
    return results

## 2.5. Layer → Demand Coefficient from `Layers_in_out`

RAMP outputs watts of **electricity**; EnergyScope demands are expressed in the **end-use layer** unit (GWh of service).  
For each layer we look up its reference electric technology in `Layers_in_out.csv` and read the `|ELECTRICITY|` input coefficient.

$$\text{demand\_GWh} = \frac{\text{ramp\_electricity\_GWh}}{|\text{coeff}|}$$

Layers mapped to `None` keep `coeff = 1.0` (electricity is already the layer).  
Mobility layers (`MOBILITY_PASSENGER`, `MOBILITY_FREIGHT`, `AVIATION_LONG_HAUL`, `SHIPPING`) are skipped — they are not derived from RAMP electricity.

In [8]:
LAYER_TO_TECH = {
    'ELECTRICITY':                   None,             # Direct, coeff=1.0
    'LIGHTING_R_C':                  'LED_BULB',
    'LIGHTING_P':                    'LED_LIGHT',
    'HEAT_HIGH_T':                   'IND_DIRECT_ELEC',
    'HEAT_LOW_T_SH':                 'DEC_DIRECT_ELEC',
    'HEAT_LOW_T_HW':                 'DEC_DIRECT_ELEC',
    'COOKING':                       'STOVE_ELEC',
    'PROCESS_COOLING':               'IND_ELEC_COLD',
    'SPACE_COOLING':                 'DEC_ELEC_COLD',
    'FOOD_PRESERVATION':             'REFRIGERATOR_EL',
    'MECHANICAL_ENERGY_COMM':        'COMM_MACHINERY_EL',
    'MECHANICAL_ENERGY_IND':         'IND_MACHINERY_EL',
    'MECHANICAL_ENERGY_MOV_AGR':     'TRACTOR_EL',
    'MECHANICAL_ENERGY_FIX_AGR':     'AGR_MACHINERY_EL',
    'MECHANICAL_ENERGY_MIN':         'MIN_MACHINERY_EL',
    'MECHANICAL_ENERGY_FISH_OTHERS': 'FISH_MACHINERY_EL',
    'NON_ENERGY':                    None,
}

MOBILITY_LAYERS = {'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'AVIATION_LONG_HAUL', 'SHIPPING'}    # no need to convert from RAMP electricity, already in layer units

# Load Layers_in_out — index = technology name, columns = layer names
lio = pd.read_csv("data/Layers_in_out.csv", sep=";", header=0, index_col=0)

# Build LAYER_TO_COEFF: end-use layer → |ELECTRICITY coefficient|
# For None entries coeff=1.0 (RAMP output already in layer units).
# Mobility layers are excluded — they are not derived from RAMP electricity.
LAYER_TO_COEFF = {}
for layer, tech in LAYER_TO_TECH.items():
    if tech is None:
        LAYER_TO_COEFF[layer] = 1.0
    else:
        LAYER_TO_COEFF[layer] = abs(float(lio.loc[tech, "ELECTRICITY"]))

print("Layer → |ELECTRICITY coefficient|:")
for layer, coeff in LAYER_TO_COEFF.items():
    print(f"  {layer:<35} {coeff:.6f}")

Layer → |ELECTRICITY coefficient|:
  ELECTRICITY                         1.000000
  LIGHTING_R_C                        2.941176
  LIGHTING_P                          2.941176
  HEAT_HIGH_T                         1.000000
  HEAT_LOW_T_SH                       1.000000
  HEAT_LOW_T_HW                       1.000000
  COOKING                             1.000000
  PROCESS_COOLING                     0.496500
  SPACE_COOLING                       0.400000
  FOOD_PRESERVATION                   2.792308
  MECHANICAL_ENERGY_COMM              1.212121
  MECHANICAL_ENERGY_IND               1.111111
  MECHANICAL_ENERGY_MOV_AGR           1.111111
  MECHANICAL_ENERGY_FIX_AGR           1.111111
  MECHANICAL_ENERGY_MIN               1.111111
  MECHANICAL_ENERGY_FISH_OTHERS       1.111111
  NON_ENERGY                          1.000000


## 2.6. Cooking Demand — Households (non-RAMP)

Household cooking demand is **not captured by RAMP** because traditional firewood and LPG stoves are non-electric appliances and were excluded from the electrical load simulation. This section quantifies that missing demand using census data and field-measured energy intensities.

### Data sources
- **Household counts by fuel**: Bolivia National Census 2024, *"NÚMERO DE VIVIENDAS SEGÚN ENERGÍA UTILIZADA PARA COCINAR"* (`CSV_final_in_excel.xlsx`)
- **Energy intensity per household**: Rural field study, Oriente/Yungas zones — best available analog for the Norte Amazónica region (SDSN Bolivia / OurWorldInData Bolivia energy profile)

### Formula (applied per stove type: wood, LPG, electric)
$$\text{GWh}_{\text{service}} = \frac{n_{\text{hh}} \times FE \;[\text{kWh/hh/year}]}{|\text{coeff}|} \div 10^6$$

`coeff` is the EnergyScope technology input coefficient from `Layers_in_out.csv` (stored as negative → absolute value taken):
- `STOVE_WOOD` → WOOD column
- `STOVE_LPG` → LPG column
- `STOVE_ELEC` → ELECTRICITY column

The result is added to **HOUSEHOLDS × COOKING** in each cluster's `Demands` table.  
**SERVICES × COOKING** (restaurant kitchen demand already in RAMP) is left untouched.

In [9]:
# Bolivia Census 2024 — households by cooking fuel, per municipality
# Source: CSV_final_in_excel.xlsx, "NÚMERO DE VIVIENDAS SEGÚN ENERGÍA UTILIZADA PARA COCINAR"
COOKING_DATA = {
    # municipality_name : (hh_wood, hh_lpg, hh_elec)
    "Ixiamas":               (1528, 1679,  1),
    "Riberalta":             (4512, 20740, 47),
    "Guayaramerín":          (1575, 8506,  19),
    "Reyes":                 (1482, 1848,  4),
    "Santa_Rosa_Beni":       (917,  1764,  6),
    "Exaltación":            (1090, 332,   1),
    "Cobija":                (300,  13727, 70),
    "Porvenir":              (223,  1950,  2),
    "Bolpebra":              (340,  452,   1),
    "Bella_Flor":            (409,  803,   2),
    "Puerto_Rico":           (405,  1513,  6),
    "San_Pedro":             (520,  85,    0),
    "Filadelfia":            (613,  1857,  3),
    "Puerto_Gonzalo_Moreno": (927,  991,   3),
    "San_Lorenzo":           (763,  1050,  1),
    "Sena":                  (1312, 1487,  2),
    "Santa_Rosa_Pando":      (235,  615,   2),
    "Ingavi":                (350,  293,   0),
    "Nueva_Esperanza":       (190,  293,   2),
    "Villa_Nueva":           (396,  302,   1),
    "Santos_Mercado":        (359,  238,   0),
}

# Final energy per household per year — rural field study Bolivia
# Source: Oriente/Yungas zones (SDSN Bolivia / OurWorldInData Bolivia energy profile)
# Values are FINAL ENERGY at the appliance; EnergyScope coefficients handle the conversion to useful service
FE_WOOD_kWh = 6760   # kWh/hh/year — Oriente field study: 66.7 MJ/day
FE_LPG_kWh  = 6970   # kWh/hh/year — Yungas field study: 68.6 MJ/day
FE_ELEC_kWh = 2900   # kWh/hh/year — modeled central value (range 2400–3500)


def compute_cooking_demand(municipality_name):
    """Return annual household cooking demand in GWh (useful service) for one municipality.

    Covers wood, LPG, and electric stoves. EnergyScope efficiency coefficients
    are read from lio (Layers_in_out.csv) — no values hardcoded here.
    """
    if municipality_name not in COOKING_DATA:
        print(f"Warning: '{municipality_name}' not found in COOKING_DATA — cooking demand set to 0.")
        return 0.0

    hh_wood, hh_lpg, hh_elec = COOKING_DATA[municipality_name]

    # Read stove coefficients from Layers_in_out (negative in CSV → absolute value)
    coeff_wood = abs(float(lio.loc["STOVE_WOOD", "WOOD"]))
    coeff_lpg  = abs(float(lio.loc["STOVE_LPG",  "LPG"]))
    coeff_elec = abs(float(lio.loc["STOVE_ELEC", "ELECTRICITY"]))

    # GWh useful service = n_hh × FE_kWh_per_hh_per_year / |coeff| / 1e6
    gwh_wood = hh_wood * FE_WOOD_kWh / coeff_wood / 1e6
    gwh_lpg  = hh_lpg  * FE_LPG_kWh  / coeff_lpg  / 1e6
    gwh_elec = hh_elec * FE_ELEC_kWh / coeff_elec / 1e6

    return gwh_wood + gwh_lpg + gwh_elec

## 3. Aggregate by cluster

Loops over each cluster, sums the RAMP demand across all its municipalities, and saves the result to `output_energyscope/C{k}/Demands_C{k}.csv`.

In [12]:
print("--- STARTING ANALYSIS ---")

for cluster_id, municipalities in CLUSTERS.items():
    df_cluster = create_empty_demands()

    total_gwh = 0.0
    by_sector = {"HOUSEHOLDS": 0.0, "SERVICES": 0.0, "INDUSTRY": 0.0, "PUBLIC_LIGHTING": 0.0}

    for muni in municipalities:
        for (sector, end_use), elec_gwh in compute_municipality(muni).items():
            if end_use in MOBILITY_LAYERS:
                continue
            coeff = LAYER_TO_COEFF.get(end_use, 1.0)
            demand_gwh = elec_gwh / coeff
            row_filter = df_cluster["parameter name"] == end_use
            df_cluster.loc[row_filter, sector] += demand_gwh
            total_gwh += demand_gwh
            if sector in by_sector:
                by_sector[sector] += demand_gwh

        # Add household cooking demand (wood + LPG + electric stoves)
        # Traditional firewood/LPG cooking is non-electric and not captured by RAMP
        cooking_gwh = compute_cooking_demand(muni)
        cooking_row = df_cluster["parameter name"] == "COOKING"
        current_cooking = df_cluster.loc[cooking_row, "HOUSEHOLDS"].fillna(0).values[0]
        df_cluster.loc[cooking_row, "HOUSEHOLDS"] = current_cooking + cooking_gwh
        total_gwh += cooking_gwh
        by_sector["HOUSEHOLDS"] += cooking_gwh

    output_file = f"{OUTPUT_DIR}/C{cluster_id}/Demands.csv"
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_cluster.to_csv(output_file, sep=";", index=False)

    print(f"Cluster {cluster_id} ({len(municipalities)} municipalities):")
    print(f"  Total Demand : {total_gwh:.4f} GWh/year")
    for sector, val in by_sector.items():
        print(f"  - {sector:<15}: {val:.4f} GWh")
    print("-" * 40)

--- STARTING ANALYSIS ---
Cluster 1 (4 municipalities):
  Total Demand : 38.3464 GWh/year
  - HOUSEHOLDS     : 37.3249 GWh
  - SERVICES       : 0.8157 GWh
  - INDUSTRY       : 0.1779 GWh
  - PUBLIC_LIGHTING: 0.0280 GWh
----------------------------------------
Cluster 2 (1 municipalities):
  Total Demand : 2.9835 GWh/year
  - HOUSEHOLDS     : 2.9046 GWh
  - SERVICES       : 0.0635 GWh
  - INDUSTRY       : 0.0133 GWh
  - PUBLIC_LIGHTING: 0.0021 GWh
----------------------------------------
Cluster 3 (3 municipalities):
  Total Demand : 165.1837 GWh/year
  - HOUSEHOLDS     : 162.3473 GWh
  - SERVICES       : 2.2719 GWh
  - INDUSTRY       : 0.4892 GWh
  - PUBLIC_LIGHTING: 0.0754 GWh
----------------------------------------
Cluster 4 (12 municipalities):
  Total Demand : 65.5529 GWh/year
  - HOUSEHOLDS     : 63.9831 GWh
  - SERVICES       : 1.2630 GWh
  - INDUSTRY       : 0.2644 GWh
  - PUBLIC_LIGHTING: 0.0424 GWh
----------------------------------------
Cluster 5 (1 municipalities):
  Total